In [ ]:
import os
import requests
from typing import Optional, Any, Dict
from dotenv import load_dotenv

def query_codestral_chat(
    prompt: str,
    model: str = "codestral-latest",  # или конкретная версия, например "codestral-2408"
    api_key: Optional[str] = None,
    return_json: bool = True,
    temperature: float = 0.7,
    max_tokens: Optional[int] = None,
) -> Optional[str]:
    """
    Отправляет запрос к Codestral через REST API напрямую.

    :param prompt: Текст запроса (сообщение от пользователя).
    :param model: Идентификатор модели (по умолчанию "codestral-latest").
    :param api_key: API-ключ Mistral. Если не указан — берётся из MISTRAL_API_KEY.
    :param return_json: Если True, добавляется response_format={"type": "json_object"}.
    :param temperature: Температура генерации (по умолчанию 0.7).
    :param max_tokens: Максимальное число токенов в ответе (опционально).
    :return: Текст ответа модели или None в случае ошибки.
    """
    if api_key is None:
        api_key = os.getenv("MISTRAL_API_KEY")
        if not api_key:
            raise ValueError("Не задан API-ключ. Установите переменную окружения MISTRAL_API_KEY.")

    url = "https://codestral.mistral.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    payload: Dict[str, Any] = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
    }

    if max_tokens is not None:
        payload["max_tokens"] = max_tokens

    if return_json:
        payload["response_format"] = {"type": "json_object"}

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при вызове Codestral API: {e}")
        if response is not None:
            print("Ответ сервера:", response.text)
        return None

In [ ]:
def get_prompt(query):
    return f"""If you are given an aggregate query, you need to modify it so that it returns all the cells needed to perform the aggregation.
    If the query does not contain any aggregation, return a single word: same
    
    Return your query as a json

    {{result: query}}

    INPUT QUERY:
    {query}
    """

In [ ]:
load_dotenv()
mistral_key = os.getenv("MISTRAL_API_KEY")
codestral_key = os.getenv("CODESTRAL_KEY")

In [ ]:
import json
from tqdm import tqdm

In [ ]:
for name in os.listdir("set"):
    for number in range(1, 11):
        try:
            with open(f"set/{name}/{number}/query.sql") as f:
                query = f.read()
        except Exception:
            with open(f"set/{name}/{number}/aggregation_query.sql") as f:
                query = f.read()
        
        print(name, number, query)

In [ ]:
for name in tqdm(os.listdir("set")):
    for number in tqdm(range(1, 11)):
        try:
            with open(f"set/{name}/{number}/query.sql") as f:
                query = f.read()
        except Exception:
            with open(f"set/{name}/{number}/aggregation_query.sql") as f:
                query = f.read()

        result = query_codestral_chat(get_prompt(query), api_key=codestral_key)
        new_query = json.loads(result)["result"]

        with open(f"set/{name}/{number}/golden_cells.sql", "w") as f:
            f.write(new_query)

In [ ]:
name = "northwind"
number = 10
try:
    with open(f"set/{name}/{number}/query.sql") as f:
        query = f.read()
except Exception:
    with open(f"set/{name}/{number}/aggregation_query.sql") as f:
        query = f.read()

result = query_codestral_chat(get_prompt(query), api_key=codestral_key)
new_query = json.loads(result)["result"]

with open(f"set/{name}/{number}/golden_cells.sql", "w") as f:
    f.write(new_query)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

with open(sys.argv[1], "r") as f:
        inputs = json.load(f)

tables_dict = {}
tables_info = {}

for table_data in inputs["tables"]:
    table_name = table_data["title"]
    df = table_to_df(table_data)
    tables_dict[table_name] = df
    tables_info[table_name] = list(df.columns)

question = inputs["question"]

llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

entities_prompt = prompt_ent_extr_from_q(question)
entities_response = llm.invoke([HumanMessage(content=entities_prompt)])
entities_str = remove_think_blocks(entities_response.content)

entities = []
if "Entities:" in entities_str:
    entities_part = entities_str.split("Entities:")[1].strip()
    try:
        if entities_part.startswith("[") and entities_part.endswith("]"):
            entities = ast.literal_eval(entities_part)
    except:
        entities = []

rel_headers_prompt = prompt_relevant_headers_multi(question, tables_info)
rel_headers_response = llm.invoke([HumanMessage(content=rel_headers_prompt)])
rel_headers_str = remove_think_blocks(rel_headers_response.content)
print(rel_headers_str)

relevant_headers = []
for line in rel_headers_str.strip().split('\n'):
    line = line.strip()
    if line:
        matches = re.findall(r'[\w\-]+\.[\w\-]+', line)
        relevant_headers.extend(matches)

mapping_prompt = prompt_entity_header_mapping_multi(question, entities, relevant_headers)
mapping_response = llm.invoke([HumanMessage(content=mapping_prompt)])
mapping_str = remove_think_blocks(mapping_response.content)
mapping_str = mapping_str.strip("```").strip("json")
print(mapping_str)

entity_header_mapping = {}

try:
    cleaned = mapping_str.strip()
    if cleaned.startswith("{") and cleaned.endswith("}"):
        parsed_dict = ast.literal_eval(cleaned)
        for entity, mapping in parsed_dict.items():
            entity_header_mapping[entity.strip('"\'')] = clean_mapping_value(str(mapping))
except:
    for line in mapping_str.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        
        for delimiter in [":", "->", "=>"]:
            if delimiter in line:
                parts = line.split(delimiter, 1)
                if len(parts) == 2:
                    entity = parts[0].strip().strip('"\'')
                    mapping = parts[1].strip().strip(',')
                    entity_header_mapping[entity] = clean_mapping_value(mapping)
                    break

print(f"Entity-Header Mapping: {entity_header_mapping}")

multi_table_graph = build_multi_table_graph(tables_dict)

# Простая визуализация
plt.figure(figsize=(8, 6))
nx.draw(G, with_labels=True, node_color='skyblue', node_size=800, font_size=15)
plt.title("Простой граф")
plt.show()